# Applied Data Science - Semester Project 2025: Machine Learning for Greek Legal Texts

In this project, we address the supervised and unsupervised classification of Greek legal documents using the Greek Legal Code dataset. We predict three levels of labels for each document: volume, chapter, and subject.

---

> Maria Schoinaki, BSc Student <br />
> Department of Informatics, Athens University of Economics and Business <br />
> p3210191@aueb.gr <br/><br/>

## 1. Data Loading and Preparation

We begin by loading the **Greek Legal Code** dataset, which contains approximately 47,000 Greek legal documents.  
Each document is labeled at three hierarchical levels:
- **Collection** (`volume`) – 47 classes
- **Chapter** (`chapter`) – 389 classes
- **Subject** (`subject`) – 2285 classes

For each classification level, we load the dataset and inspect the available splits (train, validation, test) and label structure.

In [2]:
from datasets import load_dataset

def load_dataset_with_label(label):
    """
    Loads the Greek Legal Code dataset from HuggingFace for a given label level.
    """
    dataset = load_dataset("AI-team-UoA/greek_legal_code", label)
    print("Dataset with label as", label)
    print(dataset)
    print(dataset['train'].features['label'])
    return dataset

# Volume-level classification (47 classes, default)
dataset_volume = load_dataset_with_label("volume")

# Chapter-level classification (389 classes)
dataset_chapter = load_dataset_with_label("chapter")

# Subject-level classification (2285 classes)
dataset_subject = load_dataset_with_label("subject")


Dataset with label as volume
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 28536
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 9516
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 9511
    })
})
ClassLabel(names=['ΚΟΙΝΩΝΙΚΗ ΠΡΟΝΟΙΑ', 'ΓΕΩΡΓΙΚΗ ΝΟΜΟΘΕΣΙΑ', 'ΡΑΔΙΟΦΩΝΙΑ ΚΑΙ ΤΥΠΟΣ', 'ΒΙΟΜΗΧΑΝΙΚΗ ΝΟΜΟΘΕΣΙΑ', 'ΥΓΕΙΟΝΟΜΙΚΗ ΝΟΜΟΘΕΣΙΑ', 'ΠΟΛΕΜΙΚΟ ΝΑΥΤΙΚΟ', 'ΤΑΧΥΔΡΟΜΕΙΑ - ΤΗΛΕΠΙΚΟΙΝΩΝΙΕΣ', 'ΔΑΣΗ ΚΑΙ ΚΤΗΝΟΤΡΟΦΙΑ', 'ΕΛΕΓΚΤΙΚΟ ΣΥΝΕΔΡΙΟ ΚΑΙ ΣΥΝΤΑΞΕΙΣ', 'ΠΟΛΕΜΙΚΗ ΑΕΡΟΠΟΡΙΑ', 'ΝΟΜΙΚΑ ΠΡΟΣΩΠΑ ΔΗΜΟΣΙΟΥ ΔΙΚΑΙΟΥ', 'ΝΟΜΟΘΕΣΙΑ ΑΝΩΝΥΜΩΝ ΕΤΑΙΡΕΙΩΝ ΤΡΑΠΕΖΩΝ ΚΑΙ ΧΡΗΜΑΤΙΣΤΗΡΙΩΝ', 'ΠΟΛΙΤΙΚΗ ΑΕΡΟΠΟΡΙΑ', 'ΕΜΜΕΣΗ ΦΟΡΟΛΟΓΙΑ', 'ΚΟΙΝΩΝΙΚΕΣ ΑΣΦΑΛΙΣΕΙΣ', 'ΝΟΜΟΘΕΣΙΑ ΔΗΜΩΝ ΚΑΙ ΚΟΙΝΟΤΗΤΩΝ', 'ΝΟΜΟΘΕΣΙΑ ΕΠΙΜΕΛΗΤΗΡΙΩΝ ΣΥΝΕΤΑΙΡΙΣΜΩΝ ΚΑΙ ΣΩΜΑΤΕΙΩΝ', 'ΔΗΜΟΣΙΑ ΕΡΓΑ', 'ΔΙΟΙΚΗΣΗ ΔΙΚΑΙΟΣΥΝΗΣ', 'ΑΣΦΑΛΙΣΤΙΚΑ ΤΑΜΕΙΑ', 'ΕΚΚΛΗΣΙΑΣΤΙΚΗ ΝΟΜΟΘΕΣΙΑ', 'ΕΚΠΑΙΔΕΥΤΙΚΗ ΝΟΜΟΘΕΣΙΑ', 'ΔΗΜΟΣΙΟ ΛΟΓΙΣΤΙΚΟ', 'ΤΕΛΩΝΕΙΑΚΗ ΝΟΜΟΘΕΣΙΑ', 'ΣΥΓ

## 2. Data Splitting

For each classification level (**collection**, **chapter**, **subject**), we extract the train, validation, and test splits provided in the dataset.  
We separate the input texts and corresponding labels for each split, enabling independent training, hyperparameter tuning, and final evaluation in the downstream experiments.

In [3]:
# Extract splits
def extract_splits(dataset):
    """
    Extracts train, validation, and test texts and labels from a HuggingFace dataset dict.
    Returns:
        train_texts, train_labels, val_texts, val_labels, test_texts, test_labels
    """
    train_texts = dataset['train']['text']
    train_labels = dataset['train']['label']
    
    val_texts = dataset['validation']['text']
    val_labels = dataset['validation']['label']
    
    test_texts = dataset['test']['text']
    test_labels = dataset['test']['label']
    
    return train_texts, train_labels, val_texts, val_labels, test_texts, test_labels

train_texts_volume, train_labels_volume, val_texts_volume, val_labels_volume, test_texts_volume, test_labels_volume = extract_splits(dataset_volume)

train_texts_chapter, train_labels_chapter, val_texts_chapter, val_labels_chapter, test_texts_chapter, test_labels_chapter = extract_splits(dataset_chapter)

train_texts_subject, train_labels_subject, val_texts_subject, val_labels_subject, test_texts_subject, test_labels_subject = extract_splits(dataset_subject)

In [ ]:
tasks = {
    'volume': (train_texts_volume, train_labels_volume, val_texts_volume, val_labels_volume, test_texts_volume, test_labels_volume),
    'chapter': (train_texts_chapter, train_labels_chapter, val_texts_chapter, val_labels_chapter, test_texts_chapter, test_labels_chapter),
    'subject': (train_texts_subject, train_labels_subject, val_texts_subject, val_labels_subject, test_texts_subject, test_labels_subject)
}

## 3. SVM Training and Evaluation

We define a function that automates the training and evaluation of a Support Vector Machine (SVM) classifier for text classification, using either a TF-IDF or Bag-of-Words (BoW) representation:

- The function selects and fits the appropriate vectorizer (TF-IDF or BoW) on the training data, then transforms the validation and test splits.
- A linear SVM classifier is trained using grid search to select the optimal regularization parameter C (from 0.01, 0.1, 1, 10) based on weighted F1-score on the validation set.
- The best model is evaluated on both the validation and test sets. All metrics (accuracy, precision, recall, F1) are computed using weighted averaging to account for class imbalance.
- The function prints the metrics for both validation and test sets, and returns the results for the test set (for later aggregation and reporting).

This approach ensures robust model selection and fair evaluation for both text representations and all classification levels.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import GridSearchCV

def train_evaluate_svm(
    train_texts, train_labels, val_texts, val_labels, test_texts, test_labels, 
    use_tfidf=True, max_features=20000
):
    # Select vectorizer: TF-IDF or Bag-of-Words (Count)
    Vectorizer = TfidfVectorizer if use_tfidf else CountVectorizer
    vectorizer = Vectorizer(
        max_features=max_features,
        stop_words=None,
        ngram_range=(1,2)     # (1,1) for unigrams; (1,2) also includes bigrams
    )
    
    # Fit the vectorizer on training data and transform all splits
    X_train = vectorizer.fit_transform(train_texts)
    X_val = vectorizer.transform(val_texts)
    X_test = vectorizer.transform(test_texts)
    
    # SVM classifier with Grid Search for the C hyperparameter (regularization strength)
    svm = LinearSVC()
    param_grid = {'C': [0.01, 0.1, 1, 10]}  # Logarithmic scale for efficient search
    grid = GridSearchCV(svm, param_grid, cv=3, scoring='f1_weighted', n_jobs=-1)
    grid.fit(X_train, train_labels)
    print("Best hyperparams:", grid.best_params_)
    best_svm = grid.best_estimator_
    
    # Evaluation on validation and test sets
    for split_name, X, y in [('Validation', X_val, val_labels), ('Test', X_test, test_labels)]:
        preds = best_svm.predict(X)
        acc = accuracy_score(y, preds)
        precision, recall, f1, _ = precision_recall_fscore_support(y, preds, average='weighted')
        print(f"{split_name} Results:")
        print(f"Accuracy: {acc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
        
        # Return the metrics for the test set (for summary tables)
        if split_name == 'Test':
            return {
                'accuracy': acc, 
                'precision': precision,
                'recall': recall,
                'f1': f1,
                'model': 'SVM-' + ('TFIDF' if use_tfidf else 'BoW')
            }

In [ ]:
# List to collect results for all classification tasks and vectorizer types
results = []

# Loop over each classification level (volume, chapter, subject)
for name, data in [
    ('volume', (train_texts_volume, train_labels_volume, val_texts_volume, val_labels_volume, test_texts_volume, test_labels_volume)),
    ('chapter', (train_texts_chapter, train_labels_chapter, val_texts_chapter, val_labels_chapter, test_texts_chapter, test_labels_chapter)),
    ('subject', (train_texts_subject, train_labels_subject, val_texts_subject, val_labels_subject, test_texts_subject, test_labels_subject))
]:
    # Train and evaluate SVM with TF-IDF representation
    print(f"\n--- {name.upper()} classification with TF-IDF ---")
    res = train_evaluate_svm(*data, use_tfidf=True, max_features=20000)
    res['task'] = name
    results.append(res)

    # Train and evaluate SVM with Bag-of-Words (BoW) representation
    print(f"\n--- {name.upper()} classification with BoW ---")
    res = train_evaluate_svm(*data, use_tfidf=False, max_features=20000)
    res['task'] = name
    results.append(res)

# Create a DataFrame to summarize results for all models and tasks
df_results = pd.DataFrame(results)

# Print the summary table: task, model, and all key evaluation metrics
print("\nResults:")
print(df_results[['task', 'model', 'accuracy', 'precision', 'recall', 'f1']])


--- VOLUME classification with TF-IDF ---
Best hyperparams: {'C': 1}
Validation Results:
Accuracy: 0.8348 | Precision: 0.8361 | Recall: 0.8348 | F1: 0.8340
Test Results:
Accuracy: 0.8347 | Precision: 0.8364 | Recall: 0.8347 | F1: 0.8341

--- VOLUME classification with BoW ---


c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:1235: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Best hyperparams: {'C': 0.01}
Validation Results:
Accuracy: 0.7917 | Precision: 0.7930 | Recall: 0.7917 | F1: 0.7899
Test Results:
Accuracy: 0.7946 | Precision: 0.7956 | Recall: 0.7946 | F1: 0.7932

--- CHAPTER classification with TF-IDF ---


c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


Best hyperparams: {'C': 1}


...

Best hyperparams: {'C': 0.1}


c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Validation Results:
Accuracy: 0.5740 | Precision: 0.5709 | Recall: 0.5740 | F1: 0.5494


c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test Results:
Accuracy: 0.5751 | Precision: 0.5751 | Recall: 0.5751 | F1: 0.5521

Αποτελέσματα:
      task      model  accuracy  precision    recall        f1
0   volume  SVM-TFIDF  0.834699   0.836395  0.834699  0.834062
1   volume    SVM-BoW  0.794557   0.795647  0.794557  0.793166
2  chapter  SVM-TFIDF  0.766919   0.772464  0.766919  0.760822
3  chapter    SVM-BoW  0.712169   0.721349  0.712169  0.705052
4  subject  SVM-TFIDF  0.661728   0.661324  0.661728  0.642672
5  subject    SVM-BoW  0.575137   0.575089  0.575137  0.552111


|   Task  |   Model   | Accuracy | Precision | Recall |   F1   |
| :-----: | :-------: | :------: | :-------: | :----: | :----: |
|  volume | SVM-TFIDF |  0.8347  |   0.8364  | 0.8347 | 0.8341 |
|  volume |  SVM-BoW  |  0.7946  |   0.7956  | 0.7946 | 0.7932 |
| chapter | SVM-TFIDF |  0.7669  |   0.7725  | 0.7669 | 0.7608 |
| chapter |  SVM-BoW  |  0.7122  |   0.7213  | 0.7122 | 0.7051 |
| subject | SVM-TFIDF |  0.6617  |   0.6613  | 0.6617 | 0.6427 |
| subject |  SVM-BoW  |  0.5751  |   0.5751  | 0.5751 | 0.5521 |

The results show that SVM with TF-IDF representation consistently outperforms BoW across all levels of classification (volume, chapter, subject). Both accuracy and the other evaluation metrics decrease as the number of categories increases (for example, at the subject level, where there are more than 2,000 categories).
The warning messages concern categories with very few samples, which is expected and acceptable in highly multi-class tasks with significant class imbalance.

## 4. Word2Vec + Logistic Regression Text Classification Pipeline

This code demonstrates a text classification pipeline using **Word2Vec embeddings** and a **Logistic Regression** classifier, including hyperparameter tuning via grid search and evaluation on validation and test sets.


In [5]:
import numpy as np
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def train_and_evaluate_word2vec_logreg(train_texts, train_labels, val_texts, val_labels, test_texts, test_labels, dim=300):
    # --- 1. Tokenize (εδώ με απλό split, για state-of-the-art θες tokenizer για Ελληνικά) ---
    all_texts = train_texts + val_texts + test_texts
    all_tokens = [t.split() for t in all_texts]

    # --- 2. Εκπαίδευση Word2Vec πάνω σε όλα τα tokens (μαθαίνει μόνο απ' τα δεδομένα σου!) ---
    w2v = Word2Vec(all_tokens, vector_size=dim, window=5, min_count=2, sg=1, epochs=10, workers=4)

    # --- 3. Helper για μετατροπή documents σε mean vectors ---
    def document_vector(doc):
        tokens = doc.split()
        vectors = [w2v.wv[w] for w in tokens if w in w2v.wv]
        if not vectors:
            return np.zeros(dim)
        return np.mean(vectors, axis=0)
    
    def texts_to_matrix(texts):
        return np.vstack([document_vector(t) for t in texts])
    
    # --- 4. Μετατροπή train/val/test σε embeddings ---
    X_train = texts_to_matrix(train_texts)
    X_val = texts_to_matrix(val_texts)
    X_test = texts_to_matrix(test_texts)

    # --- 5. Logistic Regression + GridSearch (για regularization C) ---
    lr = LogisticRegression(max_iter=500, solver='saga', multi_class='multinomial', n_jobs=-1)
    grid = GridSearchCV(lr, {'C':[0.01,0.1,1,10]}, cv=3, scoring='f1_weighted', n_jobs=-1)
    grid.fit(X_train, train_labels)
    best_lr = grid.best_estimator_
    print("Best hyperparams:", grid.best_params_)

    # --- 6. Evaluation ---
    for split_name, X, y in [('Validation', X_val, val_labels), ('Test', X_test, test_labels)]:
        preds = best_lr.predict(X)
        acc = accuracy_score(y, preds)
        precision, recall, f1, _ = precision_recall_fscore_support(y, preds, average='weighted')
        print(f"{split_name} Results: Accuracy={acc:.4f} | Precision={precision:.4f} | Recall={recall:.4f} | F1={f1:.4f}")

# --- Run for each label level ---
print("\n==== VOLUME classification ====")
train_and_evaluate_word2vec_logreg(train_texts_volume, train_labels_volume, val_texts_volume, val_labels_volume, test_texts_volume, test_labels_volume)

print("\n==== CHAPTER classification ====")
train_and_evaluate_word2vec_logreg(train_texts_chapter, train_labels_chapter, val_texts_chapter, val_labels_chapter, test_texts_chapter, test_labels_chapter)

print("\n==== SUBJECT classification ====")
train_and_evaluate_word2vec_logreg(train_texts_subject, train_labels_subject, val_texts_subject, val_labels_subject, test_texts_subject, test_labels_subject)



==== VOLUME classification ====


c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Best hyperparams: {'C': 10}
Validation Results: Accuracy=0.7137 | Precision=0.7131 | Recall=0.7137 | F1=0.7115
Test Results: Accuracy=0.7137 | Precision=0.7130 | Recall=0.7137 | F1=0.7114

==== CHAPTER classification ====


c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter

Best hyperparams: {'C': 10}
Validation Results: Accuracy=0.6343 | Precision=0.6438 | Recall=0.6343 | F1=0.6225
Test Results: Accuracy=0.6456 | Precision=0.6561 | Recall=0.6456 | F1=0.6364

==== SUBJECT classification ====


...

Best hyperparams: {'C': 10}


c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Validation Results: Accuracy=0.4960 | Precision=0.4874 | Recall=0.4960 | F1=0.4616


c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Test Results: Accuracy=0.4957 | Precision=0.4927 | Recall=0.4957 | F1=0.4637


| Task    | Model               | Accuracy | Precision | Recall | F1    |
|---------|---------------------|----------|-----------|--------|-------|
| volume  | Logistic-TFIDF      | 0.7137   | 0.7130    | 0.7137 | 0.7114|
| chapter | Logistic-TFIDF      | 0.6456   | 0.6561    | 0.6456 | 0.6364|
| subject | Logistic-TFIDF      | 0.4957   | 0.4927    | 0.4957 | 0.4637|


## Commentary on Logistic-TFIDF Results for Volume, Chapter, Subject

The table presents the performance of the **Logistic Regression with TF-IDF representation** on three different classification tasks: volume, chapter, and subject.

---

### Observations

- **Volume**
  - This task achieves the best performance among the three, with Accuracy, Precision, Recall, and F1 all around **0.71**.
  - This suggests the model is able to reliably identify the correct volume categories.

- **Chapter**
  - The performance drops here, with Accuracy, Precision, and Recall around **0.65**, and F1 at **0.6364**.
  - The lower scores indicate that chapter classification is more challenging, possibly due to more overlap or ambiguity between chapter categories.

- **Subject**
  - This is the most difficult task for the current model: Accuracy **0.4957**, Precision and Recall also around **0.49**, and F1 at **0.4637**.
  - The results are close to random guessing, implying that the model struggles to learn effective boundaries between subjects, possibly because the subject categories are numerous, imbalanced, or not easily separable by TF-IDF features.

---

### Overall Conclusions

- **Logistic-TFIDF** performs best in tasks with fewer and more distinct classes (like volume).
- For more complex or fine-grained tasks (like subject classification), more sophisticated models or richer representations (such as word embeddings or neural networks) may be necessary.
- The results highlight the importance of both feature selection and understanding the inherent difficulty of each classification problem.

# 5. Logistic Regression + fastText Text Classification Pipeline

This code implements a text classification pipeline using **fastText word embeddings** and **Logistic Regression**. It performs grid search to find the best regularization parameter and reports classification metrics for validation and test sets.

In [ ]:
import numpy as np
from gensim.models import FastText
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def document_vector(doc, wv, dim=300):
    tokens = doc.split()
    vectors = [wv[w] for w in tokens if w in wv]
    if not vectors:
        return np.zeros(dim)
    return np.mean(vectors, axis=0)

def texts_to_matrix(texts, wv, dim=300):
    return np.vstack([document_vector(doc, wv, dim) for doc in texts])

def run_logreg_with_fasttext(train_texts, train_labels, val_texts, val_labels, test_texts, test_labels, dim=300):
    # 1. Εκπαίδευση fastText σε όλο το corpus για καλύτερη κάλυψη λεξιλογίου
    all_texts = train_texts + val_texts + test_texts
    sentences = [t.split() for t in all_texts]
    ft = FastText(sentences, vector_size=dim, window=5, min_count=2, sg=1, epochs=10, workers=4)
    print("fastText vocabulary size:", len(ft.wv))

    # 2. Vectorization
    X_train = texts_to_matrix(train_texts, ft.wv, dim)
    X_val   = texts_to_matrix(val_texts, ft.wv, dim)
    X_test  = texts_to_matrix(test_texts, ft.wv, dim)

    # 3. Logistic Regression + GridSearch
    lr = LogisticRegression(max_iter=500, solver='saga', multi_class='multinomial', n_jobs=-1)
    grid = GridSearchCV(lr, {'C':[0.01,0.1,1,10]}, cv=3, scoring='f1_weighted', n_jobs=-1)
    grid.fit(X_train, train_labels)
    best_lr = grid.best_estimator_
    print("Best hyperparams:", grid.best_params_)

    # 4. Evaluation
    for split_name, X, y in [('Validation', X_val, val_labels), ('Test', X_test, test_labels)]:
        preds = best_lr.predict(X)
        acc = accuracy_score(y, preds)
        precision, recall, f1, _ = precision_recall_fscore_support(y, preds, average='weighted')
        print(f"{split_name} Results: Accuracy={acc:.4f} | Precision={precision:.4f} | Recall={recall:.4f} | F1={f1:.4f}")

# --- Εκτέλεσε το pipeline για κάθε label level ---
print("==== Logistic Regression + fastText για VOLUME ====")
run_logreg_with_fasttext(train_texts_volume, train_labels_volume, val_texts_volume, val_labels_volume, test_texts_volume, test_labels_volume)

print("\n==== Logistic Regression + fastText για CHAPTER ====")
run_logreg_with_fasttext(train_texts_chapter, train_labels_chapter, val_texts_chapter, val_labels_chapter, test_texts_chapter, test_labels_chapter)

print("\n==== Logistic Regression + fastText για SUBJECT ====")
run_logreg_with_fasttext(train_texts_subject, train_labels_subject, val_texts_subject, val_labels_subject, test_texts_subject, test_labels_subject)


==== Logistic Regression + fastText για VOLUME ====
fastText vocabulary size: 399018


c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Best hyperparams: {'C': 10}
Validation Results: Accuracy=0.7342 | Precision=0.7343 | Recall=0.7342 | F1=0.7302
Test Results: Accuracy=0.7294 | Precision=0.7290 | Recall=0.7294 | F1=0.7248

==== Logistic Regression + fastText για CHAPTER ====
fastText vocabulary size: 399018


c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_sag.py:349: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


Best hyperparams: {'C': 10}


c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Validation Results: Accuracy=0.5909 | Precision=0.5872 | Recall=0.5909 | F1=0.5626
Test Results: Accuracy=0.5955 | Precision=0.5994 | Recall=0.5955 | F1=0.5691

==== Logistic Regression + fastText για SUBJECT ====
fastText vocabulary size: 399018


c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


| Task    | Model            | Accuracy | Precision | Recall  | F1     |
|---------|------------------|----------|-----------|---------|--------|
| volume  | Logistic-fastText | 0.7294   | 0.7290    | 0.7294  | 0.7248 |
| chapter | Logistic-fastText | 0.5955   | 0.5994    | 0.5955  | 0.5691 |
| subject | Logistic-fastText | 0.4213   | 0.4278    | 0.4256  | 0.4298 |

## Commentary on Logistic-fastText Results

The table summarizes the performance of the **Logistic Regression + fastText** pipeline for three text classification tasks: **volume**, **chapter**, and **subject**.

---

### Observations

- **Volume**
  - The model achieves the highest performance in the volume classification task, with Accuracy, Precision, Recall, and F1 all around **0.73**.
  - This indicates that the fastText embeddings are effective in capturing the distinctions necessary for classifying volumes.

- **Chapter**
  - For the chapter task, there is a noticeable drop in all metrics (Accuracy, Precision, Recall, F1 around **0.59–0.57**).
  - This suggests that distinguishing between chapters is more challenging, possibly due to greater overlap or similarity among chapters, or more fine-grained categories.

- **Subject**
  - The subject classification task is the most challenging for this model, with all metrics dropping to the **0.42–0.43** range.
  - The results are only slightly better than random guessing, indicating that the fastText + Logistic Regression pipeline struggles with this level of granularity or with imbalanced/difficult subject categories.

---

### Overall Conclusions

- The **Logistic-fastText** model works best for the simplest/most distinct task (volume), with performance decreasing as the task becomes more fine-grained (chapter, subject).
- As complexity increases (more classes, more overlap, or smaller class sizes), the current pipeline loses effectiveness.

# 6. Naive Bayes + TF-IDF Classification Pipeline Explained

This code provides a reusable pipeline for text classification using **TF-IDF vectorization** and **Multinomial Naive Bayes** with hyperparameter search. It is organized in modular functions for flexibility and reusability across multiple tasks (volume, chapter, subject classification).

In [4]:
import numpy as np
import pandas as pd
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.model_selection import GridSearchCV
from collections import defaultdict

def vectorize_texts(train_texts, val_texts, test_texts, max_features=20000, ngram_range=(1,2)):
    # Fit on train only, transform on all
    vectorizer = TfidfVectorizer(
        max_features=max_features,
        ngram_range=ngram_range
    )
    X_train = vectorizer.fit_transform(train_texts)
    X_val = vectorizer.transform(val_texts)
    X_test = vectorizer.transform(test_texts)
    return vectorizer, X_train, X_val, X_test

def train_evaluate_nb(train_texts, train_labels, val_texts, val_labels, test_texts, test_labels, max_features=20000):
    # Vectorization
    vectorizer, X_train, X_val, X_test = vectorize_texts(train_texts, val_texts, test_texts, max_features)
    
    # Naive Bayes hyperparam search
    nb = MultinomialNB()
    param_grid = {'alpha': [0.01, 0.1, 1, 10]}
    grid = GridSearchCV(nb, param_grid, cv=3, scoring='f1_weighted', n_jobs=-1, verbose=1)
    grid.fit(X_train, train_labels)
    print("Best hyperparams:", grid.best_params_)
    best_nb = grid.best_estimator_
    
    report = {}
    for split, X, y_true in [
        ('val', X_val, val_labels),
        ('test', X_test, test_labels)
    ]:
        y_pred = best_nb.predict(X)
        acc = accuracy_score(y_true, y_pred)
        precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
        # Optionally confusion matrix (for error analysis)
        # cm = confusion_matrix(y_true, y_pred)
        print(f"{split.upper()} Results:")
        print(f"  Accuracy:  {acc:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall:    {recall:.4f}")
        print(f"  F1:        {f1:.4f}")
        report[split] = dict(accuracy=acc, precision=precision, recall=recall, f1=f1)
    report['model'] = 'NaiveBayes-TFIDF'
    return report

def evaluate_all_nb(dataset_dict, max_features=20000):
    results = []
    for name, data in dataset_dict.items():
        print(f"\n{'='*30}\n>>> {name.upper()} classification with Naive Bayes + TF-IDF")
        rep = train_evaluate_nb(*data, max_features=max_features)
        rep['task'] = name
        results.append(rep)
    # Format for report table (test metrics only)
    df_results = pd.DataFrame([
        {
            'task': r['task'],
            'model': r['model'],
            **{f"{k}_test": round(v, 4) for k, v in r['test'].items()},
            **{f"{k}_val": round(v, 4) for k, v in r['val'].items()},
        }
        for r in results
    ])
    print("\n=== FINAL TEST & VAL SCORES ===")
    print(df_results)
    return df_results

# --- Example Usage ---
dataset_dict = {
    'volume': (train_texts_volume, train_labels_volume, val_texts_volume, val_labels_volume, test_texts_volume, test_labels_volume),
    'chapter': (train_texts_chapter, train_labels_chapter, val_texts_chapter, val_labels_chapter, test_texts_chapter, test_labels_chapter),
    'subject': (train_texts_subject, train_labels_subject, val_texts_subject, val_labels_subject, test_texts_subject, test_labels_subject),
}
df_nb = evaluate_all_nb(dataset_dict, max_features=20000)



>>> VOLUME classification with Naive Bayes + TF-IDF
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best hyperparams: {'alpha': 0.01}
VAL Results:
  Accuracy:  0.7104
  Precision: 0.7233
  Recall:    0.7104
  F1:        0.7103
TEST Results:
  Accuracy:  0.7173
  Precision: 0.7326
  Recall:    0.7173
  F1:        0.7173

>>> CHAPTER classification with Naive Bayes + TF-IDF
Fitting 3 folds for each of 4 candidates, totalling 12 fits


c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


Best hyperparams: {'alpha': 0.01}
VAL Results:
  Accuracy:  0.6138
  Precision: 0.6514
  Recall:    0.6138
  F1:        0.6067
TEST Results:
  Accuracy:  0.6293
  Precision: 0.6705
  Recall:    0.6293
  F1:        0.6259

>>> SUBJECT classification with Naive Bayes + TF-IDF
Fitting 3 folds for each of 4 candidates, totalling 12 fits


c:\Users\shina\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\model_selection\_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


Best hyperparams: {'alpha': 0.01}
VAL Results:
  Accuracy:  0.4746
  Precision: 0.5131
  Recall:    0.4746
  F1:        0.4515
TEST Results:
  Accuracy:  0.4813
  Precision: 0.5203
  Recall:    0.4813
  F1:        0.4588

=== FINAL TEST & VAL SCORES ===
      task             model  accuracy_test  precision_test  recall_test  \
0   volume  NaiveBayes-TFIDF         0.7173          0.7326       0.7173   
1  chapter  NaiveBayes-TFIDF         0.6293          0.6705       0.6293   
2  subject  NaiveBayes-TFIDF         0.4813          0.5203       0.4813   

   f1_test  accuracy_val  precision_val  recall_val  f1_val  
0   0.7173        0.7104         0.7233      0.7104  0.7103  
1   0.6259        0.6138         0.6514      0.6138  0.6067  
2   0.4588        0.4746         0.5131      0.4746  0.4515  


| Task    | Model            | Accuracy | Precision | Recall |   F1   |
|---------|------------------|----------|-----------|--------|--------|
| volume  | NaiveBayes-TFIDF | 0.7173   | 0.7326    | 0.7173 | 0.7173 |
| chapter | NaiveBayes-TFIDF | 0.6293   | 0.6705    | 0.6293 | 0.6259 |
| subject | NaiveBayes-TFIDF | 0.4813   | 0.5203    | 0.4813 | 0.4588 |

# Final Results

| Task    | Model           | Accuracy | Precision | Recall  | F1     |
|---------|-----------------|----------|-----------|---------|--------|
| volume  | SVM-TFIDF       | 0.8347   | 0.8364    | 0.8347  | 0.8341 |
| volume  | SVM-BoW         | 0.7946   | 0.7956    | 0.7946  | 0.7932 |
| volume  | Logistic-fastText | 0.7294 | 0.7290    | 0.7294  | 0.7248 |
| volume  | NaiveBayes-TFIDF | 0.7173   | 0.7326    | 0.7173 | 0.7173 |
| volume  | Logistic-TFIDF  | 0.7137   | 0.7130    | 0.7137  | 0.7114 |
| chapter | SVM-TFIDF       | 0.7669   | 0.7725    | 0.7669  | 0.7608 |
| chapter | SVM-BoW         | 0.7122   | 0.7213    | 0.7122  | 0.7051 |
| chapter | Logistic-TFIDF  | 0.6456   | 0.6561    | 0.6456  | 0.6364 |
| chapter | NaiveBayes-TFIDF | 0.6293   | 0.6705    | 0.6293 | 0.6259 |
| chapter | Logistic-fastText | 0.5955 | 0.5994    | 0.5955  | 0.5691 |
| subject | SVM-TFIDF       | 0.6617   | 0.6613    | 0.6617  | 0.6427 |
| subject | SVM-BoW         | 0.5751   | 0.5751    | 0.5751  | 0.5521 |
| subject | Logistic-TFIDF  | 0.4957   | 0.4927    | 0.4957  | 0.4637 |
| subject | NaiveBayes-TFIDF | 0.4813   | 0.5203    | 0.4813 | 0.4588 |
| subject | Logistic-fastText | 0.4213 | 0.4278    | 0.4256  | 0.4298 |